# Validador de Utilizadores com Pydantic

Este notebook demonstra como utilizar a biblioteca **Pydantic** para criar modelos de dados robustos, com validações personalizadas e tratamento seguro de informações sensíveis.

## 1. Importações

Começamos por importar os módulos necessários da biblioteca padrão do Python e do Pydantic.

In [1]:
import enum
import hashlib
import re
from typing import Any

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_validator,
    model_validator,
    SecretStr,
    ValidationError,
)

## 2. Configurações e Expressões Regulares

Definimos as regras de validação para senhas e nomes utilizando expressões regulares (Regex).

In [2]:
# Expressão regular para validar senhas:
# - Pelo menos 8 caracteres, 1 minúscula, 1 maiúscula e 1 dígito.
VALID_PASSWORD_REGEX = re.compile(r"^(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{8,}$")

# Expressão regular para validar nomes:
# - Apenas letras e pelo menos 2 caracteres.
VALID_NAME_REGEX = re.compile(r"^[a-zA-Z]{2,}$")

## 3. Definição de Papéis (Roles)

Utilizamos `enum.IntFlag` para permitir que um utilizador possua múltiplos papéis simultaneamente.

In [3]:
class Role(enum.IntFlag):
    Author = 1
    Editor = 2
    Admin = 4
    SuperAdmin = 8

## 4. Modelo de Dados do Utilizador

Aqui definimos a classe `User` que herda de `BaseModel`. Incluímos validadores de campo (`field_validator`) e de modelo (`model_validator`).

**Destaques:**
- `SecretStr`: Protege a senha de ser exibida em texto limpo.
- `EmailStr`: Valida automaticamente o formato do email.
- `hashlib`: Transforma a senha em hash SHA256 antes de criar o objeto.

In [4]:
class User(BaseModel):
    name: str = Field(examples=["Arjan"])
    email: EmailStr = Field(
        examples=["user@arjancodes.com"],
        description="The email address of the user",
        frozen=True,
    )
    password: SecretStr = Field(
        examples=["Password123"], description="The password of the user"
    )
    role: Role = Field(
        default=None, description="The role of the user", examples=[1, 2, 4, 8]
    )

    @field_validator("name")
    @classmethod
    def validate_name(cls, v: str) -> str:
        if not VALID_NAME_REGEX.match(v):
            raise ValueError(
                "Name is invalid, must contain only letters and be at least 2 characters long"
            )
        return v

    @field_validator("role", mode="before")
    @classmethod
    def validate_role(cls, v: int | str | Role) -> Role:
        op = {int: lambda x: Role(x), str: lambda x: Role[x], Role: lambda x: x}
        try:
            return op[type(v)](v)
        except (KeyError, ValueError):
            raise ValueError(
                f"Role is invalid, please use one of the following: {', '.join([x.name for x in Role])}"
            )

    @model_validator(mode="before")
    @classmethod
    def validate_user(cls, v: dict[str, Any]) -> dict[str, Any]:
        if "name" not in v or "password" not in v:
            raise ValueError("Name and password are required")
        if v["name"].casefold() in v["password"].casefold():
            raise ValueError("Password cannot contain name")
        if not VALID_PASSWORD_REGEX.match(v["password"]):
            raise ValueError(
                "Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number"
            )
        v["password"] = hashlib.sha256(v["password"].encode()).hexdigest()
        return v

## 5. Funções de Suporte

Criamos uma função para encapsular a lógica de validação e capturar erros do Pydantic.

In [5]:
def validate(data: dict[str, Any]) -> None:
    try:
        user = User.model_validate(data)
        print(user)
    except ValidationError as e:
        print("User is invalid:")
        print(e)

## 6. Demonstração e Testes

Executamos vários casos de teste para verificar o comportamento do validador com dados corretos e incorretos.

In [6]:
test_data = dict(
    good_data={
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
        "role": "Admin",
    },
    bad_role={
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Password123",
        "role": "Programmer",
    },
    bad_data={
        "name": "Arjan",
        "email": "bad email",
        "password": "bad password",
    },
    bad_name={
        "name": "Arjan<-_- >",
        "email": "example@arjancodes.com",
        "password": "Password123",
    },
    duplicate={
        "name": "Arjan",
        "email": "example@arjancodes.com",
        "password": "Arjan123",
    },
    missing_data={
        "email": "<bad data>",
        "password": "<bad data>",
    },
)

for example_name, data in test_data.items():
    print(example_name)
    validate(data)
    print()

good_data
name='Arjan' email='example@arjancodes.com' password=SecretStr('**********') role=<Role.Admin: 4>

bad_role
User is invalid:
1 validation error for User
role
  Value error, Role is invalid, please use one of the following: Author, Editor, Admin, SuperAdmin [type=value_error, input_value='Programmer', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

bad_data
User is invalid:
1 validation error for User
  Value error, Password is invalid, must contain 8 characters, 1 uppercase, 1 lowercase, 1 number [type=value_error, input_value={'name': 'Arjan', 'email'...ssword': 'bad password'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error

bad_name
User is invalid:
1 validation error for User
name
  Value error, Name is invalid, must contain only letters and be at least 2 characters long [type=value_error, input_value='Arjan<-_- >', input_type=str]
    For further information visit https:/